In [ ]:
!pip install torchinfo

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchinfo import summary

In [ ]:
# ----------------------------
# Device selection (GPU/TPU/CPU)
# ----------------------------
# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
# Check for TPU availability (using PyTorch/XLA)
elif 'TPU_ACCELERATOR_TYPE' in os.environ or 'COLAB_TPU_1vm' in os.environ:
    import torch_xla
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
else:
    # Default to CPU if no GPU or TPU is available
    device = torch.device("cpu")

# Print the selected device
print(f"Using device: {device}")

In [ ]:
# Preprocessing: scale to [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    lambda x: x * 2 - 1
])

In [ ]:
# Dataset and loader
train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

In [ ]:
# Visualize real samples
data_iter = iter(train_loader)
images, labels = next(data_iter)
plt.figure(figsize=(8, 2))
for i in range(8):
    plt.subplot(1, 8, i+1)
    plt.imshow(images[i][0].cpu().numpy(), cmap='gray')
    plt.title(f"{labels[i].item()}", fontsize=10)
    plt.axis('off')
plt.suptitle("Real MNIST ([-1,1])")
plt.tight_layout()
plt.show()

In [ ]:
class ConditionalUNetScore(nn.Module):
    def __init__(self, in_ch=1, base_ch=32, num_classes=10):
        super().__init__()
        self.num_classes = num_classes
        self.class_emb = nn.Embedding(num_classes, base_ch * 4)  # inject at bottleneck

        # Encoder
        self.enc1 = self._conv_block(in_ch, base_ch)
        self.enc2 = self._conv_block(base_ch, base_ch * 2)
        self.enc3 = self._conv_block(base_ch * 2, base_ch * 4)
        self.bottleneck = self._conv_block(base_ch * 4, base_ch * 8)

        # Decoder
        self.up3 = nn.ConvTranspose2d(base_ch * 8, base_ch * 4, kernel_size=2, stride=2)
        self.dec3 = self._conv_block(base_ch * 8, base_ch * 4)
        self.up2 = nn.ConvTranspose2d(base_ch * 4, base_ch * 2, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(base_ch * 4, base_ch * 2)
        self.up1 = nn.ConvTranspose2d(base_ch * 2, base_ch, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(base_ch * 2, base_ch)
        self.final = nn.Conv2d(base_ch, in_ch, kernel_size=1)

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
            nn.ReLU()
        )

    def forward(self, x, y):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(F.avg_pool2d(e1, 2))
        e3 = self.enc3(F.avg_pool2d(e2, 2))
        b = self.bottleneck(F.avg_pool2d(e3, 2))

        # Inject class info
        emb = self.class_emb(y).view(b.shape[0], -1, 1, 1)
        b = b + emb

        # Decoder
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.final(d1)

In [ ]:
# Initialize model
model = ConditionalUNetScore(in_ch=1, base_ch=32, num_classes=10).to(device)

model = torch.compile(model)

In [ ]:
# Dummy inputs
dummy_x = torch.randn(1, 1, 28, 28).to(device)
dummy_y = torch.tensor([3]).to(device)  # class label

# Summary
summary(model, input_data=(dummy_x, dummy_y))

In [ ]:
def conditional_ssm_vr_loss(score_net, x, y):
    """
    Variance-reduced Sliced Score Matching loss with class conditioning.
    Uses Rademacher projections v ∈ {±1}^D.
    """
    B, C, H, W = x.shape
    x = x.requires_grad_(True)
    s = score_net(x, y)  # conditioned score

    # Sample Rademacher vector
    v = torch.randint(0, 2, (B, C, H, W), device=x.device, dtype=torch.float32) * 2 - 1

    # Compute v^T s
    v_dot_s = (v * s).sum(dim=[1, 2, 3])

    # Compute ∇_x (v^T s)
    grad_vdot_s = torch.autograd.grad(v_dot_s.sum(), x, create_graph=True)[0]

    # v^T (∇_x s) v
    v_hessian_v = (v * grad_vdot_s).sum(dim=[1, 2, 3])

    # ||s||^2 (not projected — variance reduction!)
    norm_s_sq = (s ** 2).sum(dim=[1, 2, 3])

    loss = (v_hessian_v + 0.5 * norm_s_sq).mean()
    return loss


In [ ]:
# Quick test
x_test = torch.randn(2, 1, 28, 28, device=device)
y_test = torch.randint(0, 10, (2,), device=device)
loss_test = conditional_ssm_vr_loss(model, x_test, y_test)
print(f"Test loss: {loss_test.item():.4f}")

In [ ]:
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# Training
epochs = 20

In [ ]:
model.train()
for epoch in range(epochs):
    total_loss = 0.0
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = conditional_ssm_vr_loss(model, x, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"✅ Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")

In [ ]:
# Save model
torch.save(model.state_dict(), "conditional_ssm_mnist.pth")
print("Model saved.")

In [ ]:
@torch.no_grad()
def conditional_langevin_dynamics(score_net, y, shape, n_steps=1000, eps=1e-3, device="cpu"):
    """
    Generate samples for class y.
    y: tensor of shape (B,) with class labels.
    """
    x = torch.randn(shape, device=device)
    score_net.eval()
    for _ in range(n_steps):
        x = x.requires_grad_()
        s = score_net(x, y)
        x = x + (eps / 2) * s + torch.sqrt(torch.tensor(eps)) * torch.randn_like(x)
        x = x.detach()
    return torch.clamp(x, -1, 1)

# Generate 8 samples per digit (0–9)
all_samples = []
all_labels = []
for digit in range(10):
    y = torch.full((8,), digit, device=device)
    samples = conditional_langevin_dynamics(model, y, (8, 1, 28, 28), n_steps=1000, eps=1e-3, device=device)
    all_samples.append(samples)
    all_labels.extend([digit] * 8)

# Concatenate
generated = torch.cat(all_samples, dim=0)
generated = (generated + 1) / 2  # [0,1] for plotting

In [ ]:
# Plot
plt.figure(figsize=(10, 10))
grid = torchvision.utils.make_grid(generated, nrow=8, padding=2)
plt.imshow(grid.cpu().permute(1, 2, 0).numpy(), cmap='gray')
plt.axis('off')
plt.title("Generated MNIST Digits (Class-Conditional SSM)")
plt.savefig("conditional_ssm_samples.png", dpi=150, bbox_inches='tight')
plt.show()